# Session 9 Assignment — Loss Functions & Output Heads

This notebook implements one observable loss harness end-to-end:

- prints every relevant tensor shape and explains each axis;
- verifies the **next-token shift with token strings**;
- masks padding and counts contributing targets;
- packs two documents and masks the cross-document boundary;
- checks untrained perplexity against vocabulary size;
- compares tied vs. untied output-head parameter counts;
- measures ordinary vs. chunked cross-entropy peak memory;
- adds a second head for `t+2`, reports both losses and their sum, and tracks training behavior.

For the most assignment-faithful memory number in Colab, select **Runtime → Change runtime type → T4 GPU**. The notebook still runs on CPU and uses an isolated RSS measurement fallback.

In [1]:
import gc, json, math, os, random, re, subprocess, sys, textwrap, time, datetime
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Code run started at {datetime.datetime.now()}")
print('PyTorch:', torch.__version__)
print('Device :', device)
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))

# Small enough to run everywhere, large enough to make the vocabulary head visible.
VOCAB_SIZE = 512
D_MODEL = 64
PAD = '[PAD]'
UNK = '[UNK]'
IGNORE_INDEX = -100
print(f'Configuration: V={VOCAB_SIZE}, D={D_MODEL}')

Code run started at 2026-08-28 18:07:25.480541
PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4
Configuration: V=512, D=64


## Tokenizer and tiny causal trunk

A tiny word/punctuation tokenizer is used so the shift can be inspected as **strings**, not integer IDs. Unused vocabulary slots are deliberately added so the vocabulary size is exactly 512. The model trunk is a small GRU because this assignment is testing the loss/output-head harness rather than attention architecture.

In [3]:
TOKEN_RE = re.compile(r"\w+|[^\w\s]")

doc_a = 'the capital of india is new delhi .'
doc_b = 'the capital of france is paris .'
doc_c = 'india is large .'
base_texts = [doc_a, doc_b, doc_c, 'A B A B .']

observed = []
for text in base_texts:
    for tok in TOKEN_RE.findall(text):
        if tok not in observed:
            observed.append(tok)

itos = [PAD, UNK] + observed
itos += [f'<extra_{i}>' for i in range(VOCAB_SIZE - len(itos))]
stoi = {tok:i for i,tok in enumerate(itos)}
PAD_ID = stoi[PAD]
UNK_ID = stoi[UNK]
A_ID, B_ID = stoi['A'], stoi['B']

assert len(itos) == VOCAB_SIZE

def encode(text):
    return [stoi.get(tok, UNK_ID) for tok in TOKEN_RE.findall(text)]

def decode_tokens(ids):
    return [itos[int(i)] for i in ids]

def pad_batch(texts):
    rows = [encode(t) for t in texts]
    T = max(map(len, rows))
    out = torch.full((len(rows), T), PAD_ID, dtype=torch.long)
    for i, row in enumerate(rows):
        out[i, :len(row)] = torch.tensor(row)
    return out

class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, tie_weights=False):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.gru = nn.GRU(d_model, d_model, batch_first=True)
        self.output_head = nn.Linear(d_model, vocab_size, bias=False)
        nn.init.normal_(self.token_embedding.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.token_embedding.weight[PAD_ID].zero_()
        if tie_weights:
            self.output_head.weight = self.token_embedding.weight
        else:
            nn.init.normal_(self.output_head.weight, mean=0.0, std=0.02)

    def forward(self, tokens):
        x = self.token_embedding(tokens)
        hidden, _ = self.gru(x)
        return hidden

model = TinyCausalLM().to(device)
output_head = model.output_head
print('Tokenizer vocabulary size:', len(itos))
print('Example tokens:', TOKEN_RE.findall(doc_a))

Tokenizer vocabulary size: 512
Example tokens: ['the', 'capital', 'of', 'india', 'is', 'new', 'delhi', '.']


# Part 1 — The observable loss harness

## 1) Print every tensor shape

In [4]:
tokens = pad_batch([doc_a, doc_b, doc_c]).to(device)
hidden = model(tokens)
logits = output_head(hidden)
shift_logits = logits[:, :-1, :]
shift_targets = tokens[:, 1:]
flat_logits = shift_logits.reshape(-1, VOCAB_SIZE)
flat_targets = shift_targets.reshape(-1)
valid_mask = flat_targets.ne(PAD_ID)

print('tokens        ', tuple(tokens.shape),       ' = [B, T]       batch, sequence positions')
print('hidden        ', tuple(hidden.shape),       ' = [B, T, D]    batch, sequence, hidden width')
print('logits        ', tuple(logits.shape),       ' = [B, T, V]    batch, sequence, vocabulary scores')
print('shift_logits  ', tuple(shift_logits.shape), ' = [B, T-1, V]  positions that predict a next token')
print('shift_targets ', tuple(shift_targets.shape),' = [B, T-1]     next-token labels')
print('flat_logits   ', tuple(flat_logits.shape),  ' = [B*(T-1), V] rows consumed by cross-entropy')
print('flat_targets  ', tuple(flat_targets.shape), ' = [B*(T-1)]    one class index per row')
print('valid_mask    ', tuple(valid_mask.shape),   ' = [B*(T-1)]    True only for contributing targets')

tokens         (3, 8)  = [B, T]       batch, sequence positions
hidden         (3, 8, 64)  = [B, T, D]    batch, sequence, hidden width
logits         (3, 8, 512)  = [B, T, V]    batch, sequence, vocabulary scores
shift_logits   (3, 7, 512)  = [B, T-1, V]  positions that predict a next token
shift_targets  (3, 7)  = [B, T-1]     next-token labels
flat_logits    (21, 512)  = [B*(T-1), V] rows consumed by cross-entropy
flat_targets   (21,)  = [B*(T-1)]    one class index per row
valid_mask     (21,)  = [B*(T-1)]    True only for contributing targets


## 2) Verify the shift with actual token strings

Every token at position `t` must predict the token at `t+1`. Printing tokens and not ids to verify.

In [5]:
row_ids = tokens[0].detach().cpu().tolist()
input_strings = decode_tokens(row_ids[:-1]) # print all elements except last one
target_strings = decode_tokens(row_ids[1:]) # print all elements starting at index 1 (second element)

# table formatting stuff that agent added
print(f"{'position':>8} | {'input token':<14} -> target token")
print('-'*48)
# actual token prints (with formatting)
for i, (x, y) in enumerate(zip(input_strings, target_strings)):
    print(f'{i:>8} | {x:<14} -> {y}')

expected = TOKEN_RE.findall(doc_a)
assert input_strings[:len(expected)-1] == expected[:-1]
assert target_strings[:len(expected)-1] == expected[1:]
print('\nSHIFT CHECK: PASS — inputs are token[t], targets are token[t+1].')

position | input token    -> target token
------------------------------------------------
       0 | the            -> capital
       1 | capital        -> of
       2 | of             -> india
       3 | india          -> is
       4 | is             -> new
       5 | new            -> delhi
       6 | delhi          -> .

SHIFT CHECK: PASS — inputs are token[t], targets are token[t+1].


## 3) Mask padding and prove the contributor count changes

Padding is ignored in the target tensor with `ignore_index=-100`. The number of loss-contributing tokens must decrease by exactly the number of padded target positions.

In [6]:
unmasked_contributors = flat_targets.numel()
masked_targets = flat_targets.clone()
masked_targets[~valid_mask] = IGNORE_INDEX
masked_contributors = masked_targets.ne(IGNORE_INDEX).sum().item()

loss_unmasked_padding = F.cross_entropy(flat_logits, flat_targets)
loss_masked_padding = F.cross_entropy(flat_logits, masked_targets, ignore_index=IGNORE_INDEX)

print('Contributors before padding mask:', unmasked_contributors)
print('Contributors after  padding mask:', masked_contributors)
print('Masked target positions         :', unmasked_contributors - masked_contributors)
print(f'Loss if PAD incorrectly contributes: {loss_unmasked_padding.item():.6f}')
print(f'Loss with PAD ignored              : {loss_masked_padding.item():.6f}')
assert masked_contributors < unmasked_contributors

Contributors before padding mask: 21
Contributors after  padding mask: 16
Masked target positions         : 5
Loss if PAD incorrectly contributes: 6.242867
Loss with PAD ignored              : 6.238751


## 4) Pack two documents and mask the boundary

When documents are concatenated, the final token of document A would otherwise be trained to predict the first token of document B. That pair is an artifact of packing, not a linguistic dependency. We remove exactly that target from the mean.

The masked mean can be higher **or** lower on a random model; masking is about correctness, not forcing a nicer number.

In [7]:
ids_a, ids_b = encode(doc_a), encode(doc_b)
packed_ids = ids_a + ids_b
packed = torch.tensor([packed_ids], dtype=torch.long, device=device)

with torch.no_grad():
    packed_hidden = model(packed)
    packed_logits = output_head(packed_hidden)[:, :-1, :]

packed_targets = packed[:, 1:].clone()
boundary_target_col = len(ids_a) - 1  # input is last token of A; target is first token of B
boundary_input = itos[packed_ids[len(ids_a)-1]]
boundary_target = itos[packed_ids[len(ids_a)]]

loss_before_boundary_mask = F.cross_entropy(
    packed_logits.reshape(-1, VOCAB_SIZE), packed_targets.reshape(-1)
)

per_token_loss = F.cross_entropy(
    packed_logits.reshape(-1, VOCAB_SIZE), packed_targets.reshape(-1), reduction='none'
).reshape_as(packed_targets)
boundary_token_loss = per_token_loss[0, boundary_target_col].item()

boundary_masked_targets = packed_targets.clone()
boundary_masked_targets[0, boundary_target_col] = IGNORE_INDEX
loss_after_boundary_mask = F.cross_entropy(
    packed_logits.reshape(-1, VOCAB_SIZE),
    boundary_masked_targets.reshape(-1),
    ignore_index=IGNORE_INDEX,
)

print('Packed document boundary:', repr(boundary_input), '->', repr(boundary_target))
print(f'Boundary token loss       : {boundary_token_loss:.6f}')
print(f'Loss before masking       : {loss_before_boundary_mask.item():.6f}')
print(f'Loss after masking        : {loss_after_boundary_mask.item():.6f}')
print('Contributors before/after :', packed_targets.numel(), '->', boundary_masked_targets.ne(IGNORE_INDEX).sum().item())
assert boundary_masked_targets.ne(IGNORE_INDEX).sum().item() == packed_targets.numel() - 1

Packed document boundary: '.' -> 'the'
Boundary token loss       : 6.232126
Loss before masking       : 6.237520
Loss after masking        : 6.237935
Contributors before/after : 14 -> 13


## 5) Compute perplexity and sanity-check the untrained model

Perplexity is `exp(mean cross-entropy)`. With small, near-zero initial logits, an untrained model is approximately uniform, so its perplexity should be near `V`.

In [8]:
with torch.no_grad():
    hidden0 = model(tokens)
    logits0 = output_head(hidden0)[:, :-1, :]
    targets0 = tokens[:, 1:].clone()
    targets0[targets0.eq(PAD_ID)] = IGNORE_INDEX
    untrained_loss = F.cross_entropy(
        logits0.reshape(-1, VOCAB_SIZE), targets0.reshape(-1), ignore_index=IGNORE_INDEX
    )
    untrained_ppl = math.exp(untrained_loss.item())

ppl_ratio_to_vocab = untrained_ppl / VOCAB_SIZE
print(f'Untrained mean CE       : {untrained_loss.item():.6f} nats')
print(f'Untrained perplexity    : {untrained_ppl:.2f}')
print(f'Vocabulary size         : {VOCAB_SIZE}')
print(f'Perplexity / vocabulary : {ppl_ratio_to_vocab:.4f}x')
assert 0.80 <= ppl_ratio_to_vocab <= 1.20, 'Sanity check failed: inspect target alignment / initialization.'
print('PERPLEXITY CHECK: PASS')

Untrained mean CE       : 6.238751 nats
Untrained perplexity    : 512.22
Vocabulary size         : 512
Perplexity / vocabulary : 1.0004x
PERPLEXITY CHECK: PASS


## 6) Tied versus untied output-head parameter counts

In a conventional token-embedding model, tying makes the output projection reuse the input embedding matrix. This saves exactly `V × D` independent parameters (bias is disabled here).

In [9]:
untied_model = TinyCausalLM(tie_weights=False)
tied_model = TinyCausalLM(tie_weights=True)

untied_total = sum(p.numel() for p in untied_model.parameters())
tied_total = sum(p.numel() for p in tied_model.parameters())
head_matrix_params = VOCAB_SIZE * D_MODEL
saved_params = untied_total - tied_total

print(f'Configuration V x D        : {VOCAB_SIZE} x {D_MODEL}')
print(f'Untied output-head params  : {head_matrix_params:,}')
print('Tied extra head params     : 0 (reuses token embedding)')
print(f'Total params, untied model : {untied_total:,}')
print(f'Total params, tied model   : {tied_total:,}')
print(f'Parameters saved by tying  : {saved_params:,}')
assert saved_params == head_matrix_params

Configuration V x D        : 512 x 64
Untied output-head params  : 32,768
Tied extra head params     : 0 (reuses token embedding)
Total params, untied model : 90,496
Total params, tied model   : 57,728
Parameters saved by tying  : 32,768


## 7) Peak memory: ordinary cross-entropy vs. chunked projection + CE

The ordinary path materializes all `[N, V]` logits at once. The chunked implementation only projects a slice of token positions at a time and immediately backpropagates its correctly weighted loss contribution. This keeps the mathematical objective exact while bounding the live logits tensor.

For CUDA we use PyTorch's allocator peak. On CPU, two fresh subprocesses independently sample RSS so allocator caching from one method does not contaminate the other.

In [10]:
MEM_N = 32768
MEM_CHUNK = 512


def full_ce_backward(hidden, head, targets):
    head.zero_grad(set_to_none=True)
    if hidden.grad is not None:
        hidden.grad = None
    logits = head(hidden)
    loss = F.cross_entropy(logits, targets)
    loss.backward()
    return float(loss.detach())


def chunked_ce_backward(hidden, head, targets, chunk_size=MEM_CHUNK):
    """Exact mean CE, but never materialize logits for more than chunk_size tokens."""
    head.zero_grad(set_to_none=True)
    if hidden.grad is not None:
        hidden.grad = None
    n = targets.numel()
    total_loss_sum = 0.0
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        chunk_logits = head(hidden[start:end])
        chunk_loss_sum = F.cross_entropy(chunk_logits, targets[start:end], reduction='sum')
        (chunk_loss_sum / n).backward()
        total_loss_sum += float(chunk_loss_sum.detach())
        del chunk_logits, chunk_loss_sum
    return total_loss_sum / n

# First, verify numerical and gradient equivalence at a smaller size.
torch.manual_seed(SEED + 1)
N_CHECK = 1024
h1 = torch.randn(N_CHECK, D_MODEL, device=device, requires_grad=True)
h2 = h1.detach().clone().requires_grad_(True)
head1 = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False).to(device)
head2 = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False).to(device)
head2.load_state_dict(head1.state_dict())
y = torch.randint(0, VOCAB_SIZE, (N_CHECK,), device=device)

loss_full_check = full_ce_backward(h1, head1, y)
loss_chunk_check = chunked_ce_backward(h2, head2, y, chunk_size=128)
max_head_grad_diff = (head1.weight.grad - head2.weight.grad).abs().max().item()
max_hidden_grad_diff = (h1.grad - h2.grad).abs().max().item()
print(f'Equivalence loss: full={loss_full_check:.8f}, chunked={loss_chunk_check:.8f}')
print(f'Max head-gradient diff  : {max_head_grad_diff:.3e}')
print(f'Max hidden-gradient diff: {max_hidden_grad_diff:.3e}')
assert abs(loss_full_check - loss_chunk_check) < 1e-5
assert max_head_grad_diff < 2e-6
assert max_hidden_grad_diff < 2e-6

del h1, h2, head1, head2, y
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

Equivalence loss: full=6.41226244, chunked=6.41226250
Max head-gradient diff  : 4.191e-09
Max hidden-gradient diff: 1.164e-10


In [11]:
def cuda_peak(method):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    torch.manual_seed(SEED + 2)
    h = torch.randn(MEM_N, D_MODEL, device='cuda', requires_grad=True)
    head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False, device='cuda')
    y = torch.randint(0, VOCAB_SIZE, (MEM_N,), device='cuda')
    torch.cuda.synchronize()
    baseline = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    if method == 'full':
        loss = full_ce_backward(h, head, y)
    else:
        loss = chunked_ce_backward(h, head, y, MEM_CHUNK)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    delta = max(0, peak - baseline)
    return loss, delta / (1024**2)


def cpu_peak_subprocess(method):
    # Fresh process per method avoids PyTorch CPU allocator reuse affecting the comparison.
    script = r"""
import gc, json, os, sys, threading, time
import psutil, torch
import torch.nn as nn
import torch.nn.functional as F
method=sys.argv[1]; N=int(sys.argv[2]); D=int(sys.argv[3]); V=int(sys.argv[4]); C=int(sys.argv[5]); seed=int(sys.argv[6])
torch.manual_seed(seed)
h=torch.randn(N,D,requires_grad=True)
head=nn.Linear(D,V,bias=False)
y=torch.randint(0,V,(N,))
proc=psutil.Process(os.getpid()); gc.collect(); baseline=proc.memory_info().rss
peak=[baseline]; stop=[False]
def sample():
    while not stop[0]:
        peak[0]=max(peak[0], proc.memory_info().rss); time.sleep(0.0005)
th=threading.Thread(target=sample,daemon=True); th.start(); time.sleep(0.005)
head.zero_grad(set_to_none=True)
if method=='full':
    logits=head(h); loss=F.cross_entropy(logits,y); loss.backward(); out=float(loss.detach())
else:
    total=0.0
    for s in range(0,N,C):
        e=min(s+C,N); logits=head(h[s:e]); ls=F.cross_entropy(logits,y[s:e],reduction='sum'); (ls/N).backward(); total+=float(ls.detach()); del logits,ls
    out=total/N
stop[0]=True; th.join(timeout=1); peak[0]=max(peak[0],proc.memory_info().rss)
print(json.dumps({'loss':out,'peak_delta_mib':max(0,peak[0]-baseline)/(1024**2)}))
"""
    cmd = [sys.executable, '-c', script, method, str(MEM_N), str(D_MODEL), str(VOCAB_SIZE), str(MEM_CHUNK), str(SEED+2)]
    raw = subprocess.check_output(cmd, text=True)
    return json.loads(raw.strip())['loss'], json.loads(raw.strip())['peak_delta_mib']

if torch.cuda.is_available():
    memory_backend = 'CUDA max_memory_allocated (increment above prepared inputs/head)'
    full_mem_loss, ordinary_peak_mib = cuda_peak('full')
    chunk_mem_loss, chunked_peak_mib = cuda_peak('chunked')
else:
    memory_backend = 'CPU RSS sampling in isolated subprocesses (increment above prepared inputs/head)'
    full_mem_loss, ordinary_peak_mib = cpu_peak_subprocess('full')
    chunk_mem_loss, chunked_peak_mib = cpu_peak_subprocess('chunked')

memory_ratio = ordinary_peak_mib / max(chunked_peak_mib, 1e-9)
print('Measurement backend:', memory_backend)
print(f'Full CE loss       : {full_mem_loss:.6f}')
print(f'Chunked CE loss    : {chunk_mem_loss:.6f}')
print(f'Ordinary peak delta: {ordinary_peak_mib:.2f} MiB')
print(f'Chunked peak delta : {chunked_peak_mib:.2f} MiB')
print(f'Peak-memory ratio  : {memory_ratio:.2f}x (ordinary / chunked)')
print(f'Max live logits shapes: full=[{MEM_N},{VOCAB_SIZE}], chunked<=[{MEM_CHUNK},{VOCAB_SIZE}]')
assert abs(full_mem_loss - chunk_mem_loss) < 1e-5

Measurement backend: CUDA max_memory_allocated (increment above prepared inputs/head)
Full CE loss       : 6.409225
Chunked CE loss    : 6.409225
Ordinary peak delta: 256.00 MiB
Chunked peak delta : 17.25 MiB
Peak-memory ratio  : 14.84x (ordinary / chunked)
Max live logits shapes: full=[32768,512], chunked<=[512,512]


# Part 2 — One extra `t+2` head

The two heads share the same causal trunk:

- **head 1** at hidden position `t` predicts token `t+1`;
- **head 2** at hidden position `t` predicts token `t+2`;
- total training loss is `L = L1 + L2`.

To make the comparison interpretable rather than dependent on memorizing a tiny paragraph, training data is generated online from a two-state Markov process. One-step transitions are more predictable than two-step transitions, so the second head has an irreducible harder target distribution. This should make `L2` remain above `L1` after learning.

In [12]:
class TinyMTP(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.gru = nn.GRU(d_model, d_model, batch_first=True)
        self.head_t1 = nn.Linear(d_model, vocab_size, bias=False)
        self.head_t2 = nn.Linear(d_model, vocab_size, bias=False)
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.head_t1.weight, std=0.02)
        nn.init.normal_(self.head_t2.weight, std=0.02)

    def hidden(self, tokens):
        h, _ = self.gru(self.token_embedding(tokens))
        return h

    def losses(self, tokens):
        h = self.hidden(tokens)
        logits1 = self.head_t1(h[:, :-1, :])
        targets1 = tokens[:, 1:]
        logits2 = self.head_t2(h[:, :-2, :])
        targets2 = tokens[:, 2:]
        l1 = F.cross_entropy(logits1.reshape(-1, VOCAB_SIZE), targets1.reshape(-1))
        l2 = F.cross_entropy(logits2.reshape(-1, VOCAB_SIZE), targets2.reshape(-1))
        return l1, l2


def markov_batch(batch_size=32, seq_len=24, p_stay=0.90, device=device):
    # states 0/1 map to the visible tokens A/B.
    states = torch.empty(batch_size, seq_len, dtype=torch.long, device=device)
    states[:, 0] = torch.randint(0, 2, (batch_size,), device=device)
    for t in range(1, seq_len):
        stay = torch.rand(batch_size, device=device) < p_stay
        states[:, t] = torch.where(stay, states[:, t-1], 1 - states[:, t-1])
    return torch.where(states.eq(0), torch.tensor(A_ID, device=device), torch.tensor(B_ID, device=device))

mtp = TinyMTP().to(device)
opt = torch.optim.AdamW(mtp.parameters(), lr=3e-3, weight_decay=0.0)

# Initial losses on a fixed validation sample.
torch.manual_seed(SEED + 10)
val_tokens = markov_batch(batch_size=256, seq_len=24, p_stay=0.90)
with torch.no_grad():
    initial_l1, initial_l2 = mtp.losses(val_tokens)

history=[]
TRAIN_STEPS=1000
for step in range(1, TRAIN_STEPS+1):
    train_tokens = markov_batch(batch_size=32, seq_len=24, p_stay=0.90)
    l1, l2 = mtp.losses(train_tokens)
    total = l1 + l2
    opt.zero_grad(set_to_none=True)
    total.backward()
    torch.nn.utils.clip_grad_norm_(mtp.parameters(), 1.0)
    opt.step()
    if step == 1 or step % 50 == 0:
        with torch.no_grad():
            v1, v2 = mtp.losses(val_tokens)
        history.append((step, v1.item(), v2.item(), (v1+v2).item()))
        print(f'step {step:3d} | L1(t+1)={v1.item():.4f} | L2(t+2)={v2.item():.4f} | sum={v1.item()+v2.item():.4f}')

with torch.no_grad():
    final_l1_t, final_l2_t = mtp.losses(val_tokens)
part2_loss_t1 = final_l1_t.item()
part2_loss_t2 = final_l2_t.item()
part2_loss_sum = part2_loss_t1 + part2_loss_t2

print('\nInitial validation losses:')
print(f'  t+1: {initial_l1.item():.6f}')
print(f'  t+2: {initial_l2.item():.6f}')
print('Final validation losses:')
print(f'  t+1: {part2_loss_t1:.6f}')
print(f'  t+2: {part2_loss_t2:.6f}')
print(f'  sum: {part2_loss_sum:.6f}')
print(f'  gap L2-L1: {part2_loss_t2-part2_loss_t1:.6f}')

step   1 | L1(t+1)=6.2082 | L2(t+2)=6.1946 | sum=12.4029
step  50 | L1(t+1)=0.6957 | L2(t+2)=0.6956 | sum=1.3913
step 100 | L1(t+1)=0.7102 | L2(t+2)=0.7097 | sum=1.4199
step 150 | L1(t+1)=0.5079 | L2(t+2)=0.5622 | sum=1.0701
step 200 | L1(t+1)=0.3604 | L2(t+2)=0.4721 | sum=0.8325
step 250 | L1(t+1)=0.3227 | L2(t+2)=0.4649 | sum=0.7876
step 300 | L1(t+1)=0.3197 | L2(t+2)=0.4646 | sum=0.7844
step 350 | L1(t+1)=0.3211 | L2(t+2)=0.4692 | sum=0.7903
step 400 | L1(t+1)=0.3184 | L2(t+2)=0.4655 | sum=0.7839
step 450 | L1(t+1)=0.3189 | L2(t+2)=0.4687 | sum=0.7877
step 500 | L1(t+1)=0.3186 | L2(t+2)=0.4678 | sum=0.7864
step 550 | L1(t+1)=0.3186 | L2(t+2)=0.4645 | sum=0.7830
step 600 | L1(t+1)=0.3205 | L2(t+2)=0.4656 | sum=0.7861
step 650 | L1(t+1)=0.3184 | L2(t+2)=0.4647 | sum=0.7832
step 700 | L1(t+1)=0.3186 | L2(t+2)=0.4643 | sum=0.7829
step 750 | L1(t+1)=0.3185 | L2(t+2)=0.4643 | sum=0.7828
step 800 | L1(t+1)=0.3205 | L2(t+2)=0.4672 | sum=0.7876
step 850 | L1(t+1)=0.3189 | L2(t+2)=0.4644 | su

In [13]:
# Theoretical entropy check for the data-generating process.
p = 0.90
p2_same = p*p + (1-p)*(1-p)
H1 = -(p*math.log(p) + (1-p)*math.log(1-p))
H2 = -(p2_same*math.log(p2_same) + (1-p2_same)*math.log(1-p2_same))
print(f'One-step conditional entropy H1: {H1:.4f} nats')
print(f'Two-step conditional entropy H2: {H2:.4f} nats')
print('H2 > H1:', H2 > H1)
print('\nInterpretation: the t+2 target compounds transition uncertainty, so its best-achievable')
print('cross-entropy is higher. The observed second-head loss should therefore settle above')
print('the first-head loss rather than matching it.')

One-step conditional entropy H1: 0.3251 nats
Two-step conditional entropy H2: 0.4714 nats
H2 > H1: True

Interpretation: the t+2 target compounds transition uncertainty, so its best-achievable
cross-entropy is higher. The observed second-head loss should therefore settle above
the first-head loss rather than matching it.


# Final submission summary

The cell below prints a compact Markdown block suitable for the GitHub README and writes the machine-readable results to `session9_results.json`.

In [15]:
results = {
    'configuration': {'vocab_size': VOCAB_SIZE, 'd_model': D_MODEL, 'device': str(device)},
    'part1': {
        'shape_tokens': list(tokens.shape),
        'shape_hidden': list(hidden.shape),
        'shape_logits': list(logits.shape),
        'shift_verified': True,
        'padding_contributors_before': int(unmasked_contributors),
        'padding_contributors_after': int(masked_contributors),
        'boundary_loss_before': float(loss_before_boundary_mask.item()),
        'boundary_loss_after': float(loss_after_boundary_mask.item()),
        'untrained_loss': float(untrained_loss.item()),
        'untrained_perplexity': float(untrained_ppl),
        'ppl_over_vocab': float(ppl_ratio_to_vocab),
        'untied_total_params': int(untied_total),
        'tied_total_params': int(tied_total),
        'saved_by_tying': int(saved_params),
        'memory_backend': memory_backend,
        'ordinary_peak_mib': float(ordinary_peak_mib),
        'chunked_peak_mib': float(chunked_peak_mib),
        'ordinary_over_chunked_ratio': float(memory_ratio),
    },
    'part2': {
        'loss_t1': float(part2_loss_t1),
        'loss_t2': float(part2_loss_t2),
        'loss_sum': float(part2_loss_sum),
        'loss_gap_t2_minus_t1': float(part2_loss_t2-part2_loss_t1),
    }
}
with open('session9_results.json','w') as f:
    json.dump(results, f, indent=2)

summary = f"""## Session 9 results ({device})

1. **Shapes:** tokens `{tuple(tokens.shape)}`, hidden `{tuple(hidden.shape)}`, logits `{tuple(logits.shape)}`; shifted logits `{tuple(shift_logits.shape)}` and targets `{tuple(shift_targets.shape)}`.
2. **Shift:** verified with printed token strings (`token[t] -> token[t+1]`).
3. **Padding:** contributing targets changed **{unmasked_contributors} -> {masked_contributors}** after masking PAD.
4. **Packed boundary:** loss **{loss_before_boundary_mask.item():.6f} -> {loss_after_boundary_mask.item():.6f}** after removing the single cross-document target.
5. **Untrained perplexity:** **{untrained_ppl:.2f}** for vocabulary **{VOCAB_SIZE}** ({ppl_ratio_to_vocab:.4f}x V).
6. **Weight tying:** untied total **{untied_total:,}** vs tied total **{tied_total:,}**; saves **{saved_params:,} = V x D** parameters.
7. **Peak memory:** ordinary **{ordinary_peak_mib:.2f} MiB**, chunked **{chunked_peak_mib:.2f} MiB**, ratio **{memory_ratio:.2f}x** using {memory_backend}.

**Part 2:** `t+1` loss **{part2_loss_t1:.6f}**, `t+2` loss **{part2_loss_t2:.6f}**, sum **{part2_loss_sum:.6f}**. The second head remains harder because two-step uncertainty is higher than one-step uncertainty in the online Markov data.
"""
print(summary)

## Session 9 results (cuda)

1. **Shapes:** tokens `(3, 8)`, hidden `(3, 8, 64)`, logits `(3, 8, 512)`; shifted logits `(3, 7, 512)` and targets `(3, 7)`.
2. **Shift:** verified with printed token strings (`token[t] -> token[t+1]`).
3. **Padding:** contributing targets changed **21 -> 16** after masking PAD.
4. **Packed boundary:** loss **6.237520 -> 6.237935** after removing the single cross-document target.
5. **Untrained perplexity:** **512.22** for vocabulary **512** (1.0004x V).
6. **Weight tying:** untied total **90,496** vs tied total **57,728**; saves **32,768 = V x D** parameters.
7. **Peak memory:** ordinary **256.00 MiB**, chunked **17.25 MiB**, ratio **14.84x** using CUDA max_memory_allocated (increment above prepared inputs/head).

**Part 2:** `t+1` loss **0.318380**, `t+2` loss **0.464809**, sum **0.783189**. The second head remains harder because two-step uncertainty is higher than one-step uncertainty in the online Markov data.

